# DotWall cost transfer across wall layouts

Render each frame directly and train on geodesic ordering. No world model or planner rollout is used. Run setup and the small smoke check first. Then set `RUN_FULL=True` and run the four stages in order. Drive receives reports and checkpoints; frame tokens stay on the Colab VM.


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,shutil,signal,subprocess,sys,time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime, then Run all.'
DRIVE=Path('/content/drive/MyDrive')
RESUME_RUN_ID=None
RUN_ID=RESUME_RUN_ID or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=DRIVE/'datasets/curriculum_cost_adapter/dotwall_layout_transfer_v1'/RUN_ID
REPO=Path('/content/i-jepa-wm-layout-cost')
REPO_URL='https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT='cd9a3f5de874d9ca16c231dca8f7a628ef44ece4'
DINO_ROOT=Path('/content/dino_wm')
DINO_COMMIT='0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER_DRIVE=DRIVE/'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
ARTIFACTS=Path('/content/dotwall-layout-cost-artifacts')
ENCODER=ARTIFACTS/'encoder.pth.tar'
ENCODER_SHA='3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()

def shell(command,cwd=None):
    command=list(map(str,command));print('$',' '.join(command),flush=True)
    with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            for line in process.stdout:print(line,end='',flush=True)
            code=process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.terminate()
            raise
    if code:raise subprocess.CalledProcessError(code,command)

print('GPU:',torch.cuda.get_device_name())
print('Run ID:',RUN_ID)
print('Output:',OUTPUT)
CACHE=Path('/content/dotwall-layout-cost-cache')/RUN_ID
MODULE_SHA='e6f0ae2b587c646d37bb48886c8f6fb93f0f647d86b9cce82c123fb2d74150c7'
print('VM cache:',CACHE)


In [ ]:
if not REPO.exists():shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
else:
    assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()==REPO_URL
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Clone has tracked changes.'
    shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pandas','pillow','imageio','imageio-ffmpeg'])
if not DINO_ROOT.exists():shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Renderer has tracked changes.'
shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
ARTIFACTS.mkdir(parents=True,exist_ok=True)
for source,target,expected in [(ENCODER_DRIVE,ENCODER,ENCODER_SHA)]:
    assert source.is_file(), f'Missing frozen artifact: {source}'
    if not target.exists() or sha256(target)!=expected:shutil.copy2(source,target)
    assert sha256(target)==expected, f'Unexpected checkpoint: {source}'

## Install the tested runner

The next cell carries the new runner source inside this notebook, so the pinned repository commit is sufficient.


In [ ]:
%%writefile /content/i-jepa-wm-layout-cost/src/mylonas_ijepa_wm/dotwall_layout_cost.py
"""Rendered DotWall square-curriculum cost transfer across wall layouts.

The frozen encoder and WallAdapter are supplied by the caller. Frame/token
caches stay on the VM; reports and adapter checkpoints can live on Drive.
"""
from itertools import combinations
from pathlib import Path
import json

import numpy as np
import torch
from scipy.stats import spearmanr

from src.mylonas_ijepa_wm.curriculum_protocol import digest, write_json
from src.mylonas_ijepa_wm.curriculum_replay import load_adapter, save_adapter
from src.mylonas_ijepa_wm.multigoal_ranking import GAP, fit
from src.mylonas_ijepa_wm.ranking_planning import _frame_hash
from src.mylonas_ijepa_wm.wall_layout_manifest import TRAIN_LAYOUTS, TEST_LAYOUTS


STANDARD = (32, 30)
BANDS = tuple(zip((0, 6, 12, 18, 24, 30, 36, 42), (6, 12, 18, 24, 30, 36, 42, 46)))
TRAIN_GOALS = ((12, 14), (12, 48), (52, 14), (52, 48))
EVAL_GOALS = ((14, 22), (14, 40), (50, 22), (50, 40))
EVAL_AXIS = tuple(range(5, 60, 2))
ALL_LAYOUTS = tuple(map(tuple, TRAIN_LAYOUTS + TEST_LAYOUTS))
TIE = 1e-7


def layout_id(layout):
    return f"wall{int(layout[0]):02d}_door{int(layout[1]):02d}"


def valid(point, layout):
    x, y = map(float, point)
    wall_x, door_y = map(float, layout)
    return (4.001 <= x <= 59.999 and 4.001 <= y <= 59.999 and
            (x < wall_x - 3 or x > wall_x + 3 or door_y - 4 < y < door_y + 4))


def _visible(a, b, layout):
    left, right = layout[0] - 3, layout[0] + 3
    low, high = layout[1] - 4, layout[1] + 4
    dx, dy = b - a
    if abs(dx) < 1e-12:
        return not left < a[0] < right or (min(a[1], b[1]) >= low and max(a[1], b[1]) <= high)
    entry, leave = sorted(((left - a[0]) / dx, (right - a[0]) / dx))
    lo, hi = max(0., entry), min(1., leave)
    if lo >= hi:
        return True
    y1, y2 = a[1] + lo * dy, a[1] + hi * dy
    return min(y1, y2) >= low - 1e-9 and max(y1, y2) <= high + 1e-9


def distance(point, goal, layout):
    """Ball-center shortest path around two solid wall rectangles."""
    if not valid(point, layout) or not valid(goal, layout):
        raise ValueError("geodesic endpoints must be legal in their layout")
    left, right = layout[0] - 3, layout[0] + 3
    low, high = layout[1] - 4, layout[1] + 4
    points = np.asarray([point, goal, (left, low), (left, high),
                         (right, low), (right, high)], dtype=float)
    if _visible(points[0], points[1], layout):
        return float(np.linalg.norm(points[0] - points[1]))
    lengths = np.full((6, 6), np.inf)
    np.fill_diagonal(lengths, 0)
    for i, j in combinations(range(6), 2):
        if _visible(points[i], points[j], layout):
            lengths[i, j] = lengths[j, i] = np.linalg.norm(points[i] - points[j])
    for k in range(6):
        lengths = np.minimum(lengths, lengths[:, k, None] + lengths[None, k, :])
    return float(lengths[0, 1])


def make_design(layout, *, include_training=True, bands=BANDS, eval_axis=EVAL_AXIS):
    """One layout's distinct frames, cumulative fit bands, and offset eval grid."""
    layout = tuple(layout)
    points, lookup = [], {}

    def intern(point):
        key = tuple(map(float, point))
        if key not in lookup:
            lookup[key] = len(points)
            points.append(list(key))
        return lookup[key]

    train_goals = [intern(g) for g in TRAIN_GOALS] if include_training else []
    eval_goals = [intern(g) for g in EVAL_GOALS]
    protected = set(train_goals + eval_goals)
    samples = {}
    if include_training:
        for goal_id in train_goals:
            gx, gy = points[goal_id]
            nodes = []
            for band, (inner, outer) in enumerate(bands):
                for dy in range(-outer, outer + 1, 2):
                    for dx in range(-outer, outer + 1, 2):
                        extent = max(abs(dx), abs(dy))
                        point = (gx + dx, gy + dy)
                        if inner < extent <= outer and valid(point, layout):
                            node_id = intern(point)
                            if node_id not in protected:
                                nodes.append(dict(id=node_id, band=band,
                                                  distance=distance(point, (gx, gy), layout)))
            samples[goal_id] = nodes
    grid = [intern((x, y)) if valid((x, y), layout) else None
            for y in eval_axis for x in eval_axis]
    return dict(layout=list(layout), points=points, train_goals=train_goals,
                eval_goals=eval_goals, samples=samples, grid=grid,
                axis=list(eval_axis), bands=[list(b) for b in bands])


def _triples(design, hashes, stage, offset):
    """All eligible same-goal comparisons, omitting rendered frame aliases."""
    protected = {hashes[i] for i in design["train_goals"] + design["eval_goals"]}
    result = []
    for goal_id in design["train_goals"]:
        nodes, used = [], set(protected)
        for node in design["samples"][goal_id]:
            frame_hash = hashes[node["id"]]
            if node["band"] <= stage and frame_hash not in used:
                nodes.append(node)
                used.add(frame_hash)
        if len(nodes) < 2:
            continue
        ids = np.asarray([n["id"] + offset for n in nodes], dtype=np.int32)
        distances = np.asarray([n["distance"] for n in nodes])
        a, b = np.triu_indices(len(nodes), 1)
        delta = distances[a] - distances[b]
        eligible = np.abs(delta) >= GAP
        a, b, delta = a[eligible], b[eligible], delta[eligible]
        far = np.where(delta > 0, ids[a], ids[b])
        near = np.where(delta > 0, ids[b], ids[a])
        result.append(np.column_stack((np.full(len(far), goal_id + offset), far, near)))
    return np.concatenate(result).astype(np.int32) if result else np.empty((0, 3), dtype=np.int32)


def balanced_schedule(buckets, updates, batch_size, *, seed):
    """Choose a layout, then a goal, then a comparison for each draw."""
    layouts = sorted(buckets)
    if not layouts or updates < 1 or batch_size < 1:
        raise ValueError("invalid balanced schedule")
    rng = np.random.default_rng(seed)
    schedule = np.empty((updates, batch_size), dtype=np.int64)
    counts = {layout: 0 for layout in layouts}
    for row in schedule:
        for j in range(batch_size):
            layout = layouts[int(rng.integers(len(layouts)))]
            goals = sorted(buckets[layout])
            if not goals:
                raise ValueError("layout has no eligible goals")
            goal = goals[int(rng.integers(len(goals)))]
            candidates = buckets[layout][goal]
            if not len(candidates):
                raise ValueError("goal has no eligible comparisons")
            row[j] = candidates[int(rng.integers(len(candidates)))]
            counts[layout] += 1
    return schedule, counts


def _encode_cache(cache_dir, env, encode, designs, runtime):
    """Cache float16 patch tokens and frame hashes on VM disk."""
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    offsets, total = {}, 0
    for key, design in designs.items():
        offsets[key] = total
        total += len(design["points"])
    first = next(iter(designs.values()))
    env.reset(0, layout=first["layout"], start=TRAIN_GOALS[0], goal=EVAL_GOALS[0])
    probe = encode(np.stack([env.render_goal()])).detach().cpu()
    shape = (total, *probe.shape[1:])
    path = cache_dir / "tokens.npy"
    hashes_path = cache_dir / "hashes.json"
    signature = digest(dict(runtime=runtime, designs=designs))
    marker = cache_dir / "cache.json"
    config = dict(design=signature, shape=list(shape), dtype="float16")
    config_path = cache_dir / "cache_config.json"
    if config_path.exists():
        if json.loads(config_path.read_text()) != config:
            raise ValueError("VM token cache belongs to a different design or runtime")
        tokens = np.load(path, mmap_mode="r+")
        if tokens.shape != shape or tokens.dtype != np.float16:
            raise ValueError("VM token cache shape or dtype differs")
    else:
        tokens = np.lib.format.open_memmap(path, mode="w+", dtype=np.float16, shape=shape)
        write_json(config_path, config)
    if marker.exists():
        saved = json.loads(marker.read_text())
        if saved != config:
            raise ValueError("VM token cache belongs to a different design")
        return torch.from_numpy(tokens), json.loads(hashes_path.read_text()), offsets
    hashes = {}
    for key, design in designs.items():
        layout_hash_path = cache_dir / f"{key}.hashes.json"
        layout_marker = cache_dir / f"{key}.complete.json"
        if layout_marker.exists():
            local_hashes = json.loads(layout_hash_path.read_text())
            expected = dict(design=signature, count=len(design["points"]), hashes=digest(local_hashes))
            if json.loads(layout_marker.read_text()) != expected:
                raise ValueError(f"completed layout cache differs: {key}")
            hashes[key] = local_hashes
            print(f"Reused {key}: {len(local_hashes)} rendered frames", flush=True)
            continue
        layout = design["layout"]
        env.reset(0, layout=layout, start=TRAIN_GOALS[0], goal=EVAL_GOALS[0])
        if _frame_hash(env.render_goal()) != _frame_hash(env._render_at(EVAL_GOALS[0])):
            raise ValueError("goal render differs from the position render")
        local_hashes = []
        for start in range(0, len(design["points"]), 8):
            frames = np.stack([env._render_at(p) for p in design["points"][start:start + 8]])
            local_hashes.extend(map(_frame_hash, frames))
            batch = encode(frames).detach().cpu().numpy().astype(np.float16)
            tokens[offsets[key] + start:offsets[key] + start + len(batch)] = batch
        hashes[key] = local_hashes
        tokens.flush()
        write_json(layout_hash_path, local_hashes)
        write_json(layout_marker, dict(design=signature, count=len(local_hashes),
                                       hashes=digest(local_hashes)))
        print(f"Encoded {key}: {len(local_hashes)} rendered frames", flush=True)
    # Marker is written last, so an interrupted cache is never treated as complete.
    write_json(hashes_path, hashes)
    write_json(marker, config)
    return torch.from_numpy(tokens), hashes, offsets


def prepare(output, cache_dir, env, encode, *, runtime, smoke=False):
    """Render each layout directly and build a restartable VM token cache."""
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    layouts = (STANDARD, tuple(TRAIN_LAYOUTS[0])) if smoke else ALL_LAYOUTS
    bands = BANDS[:1] if smoke else BANDS
    axis = EVAL_AXIS[::4] if smoke else EVAL_AXIS
    designs = {layout_id(layout): make_design(layout, include_training=layout in map(tuple, TRAIN_LAYOUTS),
                                               bands=bands, eval_axis=axis)
               for layout in layouts}
    manifest = dict(schema_version=1, train_layouts=[list(x) for x in TRAIN_LAYOUTS],
                    unseen_layouts=[list(x) for x in TEST_LAYOUTS],
                    train_goals=[list(x) for x in TRAIN_GOALS], eval_goals=[list(x) for x in EVAL_GOALS],
                    bands=[list(x) for x in bands], eval_axis=list(axis), smoke=smoke,
                    runtime=runtime)
    manifest_path = output / "layout_goal_manifest.json"
    if manifest_path.exists() and json.loads(manifest_path.read_text()) != manifest:
        raise ValueError("Drive run belongs to a different layout or runtime manifest")
    write_json(manifest_path, manifest)
    tokens, hashes, offsets = _encode_cache(cache_dir, env, encode, designs, runtime)
    hash_manifest = {key: digest(rows) for key, rows in hashes.items()}
    hash_path = output / "frame_hash_manifest.json"
    if hash_path.exists() and json.loads(hash_path.read_text()) != hash_manifest:
        raise ValueError("rendered frame hashes changed for this Drive run")
    write_json(hash_path, hash_manifest)
    return dict(designs=designs, tokens=tokens, hashes=hashes, offsets=offsets,
                output=output, smoke=smoke, runtime=runtime)


def _raw_scale(tokens, triples):
    """Exact median positive raw MSE over unique training goal/query exposures."""
    values = []
    for goal in np.unique(triples[:, 0]):
        selected = triples[triples[:, 0] == goal]
        queries = np.unique(selected[:, 1:])
        for group in np.array_split(queries, max(1, (len(queries) + 31) // 32)):
            values.extend(((tokens[group].float() - tokens[int(goal)].float())
                           .square().mean((1, 2))).tolist())
    positive = np.asarray(values)
    positive = positive[positive > 0]
    if not len(positive):
        raise ValueError("raw training cost scale is zero")
    return float(np.median(positive))


def fit_model(study, which, *, updates=500, batch_size=16, device="cuda"):
    """Fit eight cumulative checkpoints, balanced by goal or layout and goal."""
    if which not in ("fixed", "multi"):
        raise ValueError("fit kind must be fixed or multi")
    keys = [layout_id(STANDARD)] if which == "fixed" else [layout_id(x) for x in TRAIN_LAYOUTS]
    if study["smoke"] and which == "multi":
        keys = [layout_id(STANDARD), layout_id(TRAIN_LAYOUTS[0])]
    folder = study["output"] / which
    folder.mkdir(parents=True, exist_ok=True)
    previous = None
    stages = 1 if study["smoke"] else len(BANDS)
    for stage in range(stages):
        parts, buckets, base = [], {}, 0
        for key in keys:
            triples = _triples(study["designs"][key], study["hashes"][key], stage, study["offsets"][key])
            if not len(triples):
                raise ValueError(f"no fitting pairs for {key} at band {stage + 1}")
            parts.append(triples)
            layout_groups = {}
            for goal in np.unique(triples[:, 0]):
                layout_groups[int(goal)] = base + np.flatnonzero(triples[:, 0] == goal)
            buckets[key] = layout_groups
            base += len(triples)
        triples = np.concatenate(parts)
        schedule, counts = balanced_schedule(buckets, updates, batch_size, seed=stage)
        checkpoint = folder / f"band_{stage + 1:02d}.pt"
        provenance = dict(runtime_id=digest(dict(which=which, layouts=keys,
                                                 design=[digest(study["designs"][key]) for key in keys],
                                                 updates=updates, batch_size=batch_size,
                                                 runtime=study["runtime"])),
                          band=stage + 1)
        if checkpoint.exists():
            previous, payload = load_adapter(checkpoint, "cpu", expected_runtime_id=provenance["runtime_id"])
            if payload["provenance"] != provenance:
                raise ValueError("checkpoint band provenance differs")
            training = payload["training"]
        else:
            scale = _raw_scale(study["tokens"], triples)
            previous, training = fit(study["tokens"], triples, scale,
                                     seed=stage, updates=updates, batch_size=batch_size,
                                     device=device, initial_adapter=previous,
                                     sample_schedule=schedule,
                                     progress=lambda row: print(which, stage + 1, row, flush=True))
            save_adapter(checkpoint, previous, training, provenance)
        write_json(folder / f"band_{stage + 1:02d}.json",
                   dict(band=stage + 1, half_width=BANDS[stage][1],
                        eligible_pairs=len(triples), draws_per_layout=counts,
                        draws_per_goal={key: {str(g): int(np.isin(schedule, ids).sum())
                                              for g, ids in buckets[key].items()} for key in keys},
                        training={k: v for k, v in training.items() if k not in
                                  ("history", "sampled_triplet_ids", "endpoint_counts", "reference_counts",
                                   "far_counts", "near_counts")}))
        print(f"{which} band {stage + 1}: {len(triples)} pairs, {counts}", flush=True)
    return folder / f"band_{stages:02d}.pt"


@torch.no_grad()
def _scores(adapter, tokens, goal_index, query_indices, device):
    goal = tokens[goal_index:goal_index + 1].to(device).float()
    if adapter is not None:
        goal = adapter(goal)
    values = []
    for group in np.array_split(np.asarray(query_indices, dtype=int),
                                max(1, (len(query_indices) + 15) // 16)):
        query = tokens[group].to(device).float()
        if adapter is not None:
            query = adapter(query)
        values.extend((query - goal).square().mean((1, 2)).cpu().tolist())
    return np.asarray(values)


def _rank_metric(truth, cost):
    farther, nearer = np.where(truth[:, None] - truth[None, :] >= GAP)
    delta = cost[farther] - cost[nearer]
    tolerance = max(1e-7, .001 * float(np.ptp(cost)))
    return dict(pairs=len(delta), correct=int((delta > tolerance).sum()),
                accuracy=float((delta > tolerance).mean()) if len(delta) else None,
                spearman=float(spearmanr(truth, cost).statistic) if len(truth) > 2 else None)


def evaluate(study, keys, checkpoints, name, *, device="cuda"):
    """Score exactly one layout/goal/query mask for all available models."""
    models = {"raw": None}
    for label, path in checkpoints.items():
        models[label] = load_adapter(path, device)[0]
    output = study["output"] / name
    output.mkdir(parents=True, exist_ok=True)
    all_rows = []
    for key in keys:
        design = study["designs"][key]
        layout = design["layout"]
        seen_hashes = {study["hashes"][key][i] for i in design["train_goals"]}
        for nodes in design["samples"].values():
            seen_hashes.update(study["hashes"][key][n["id"]] for n in nodes)
        grid_ids = [i for i in design["grid"] if i is not None
                    and study["hashes"][key][i] not in seen_hashes]
        positions = np.asarray([design["points"][i] for i in grid_ids])
        offset = study["offsets"][key]
        for goal_number, goal_id in enumerate(design["eval_goals"]):
            goal = design["points"][goal_id]
            truth = np.asarray([distance(p, goal, layout) for p in positions])
            scores = {label: _scores(model, study["tokens"], offset + goal_id,
                                     offset + np.asarray(grid_ids), device)
                      for label, model in models.items()}
            self_cost = {label: float(_scores(model, study["tokens"], offset + goal_id,
                                             [offset + goal_id], device)[0])
                         for label, model in models.items()}
            if any(value > 1e-8 for value in self_cost.values()):
                raise ValueError("goal frame did not score near zero against itself")
            left, right = layout[0] - 3, layout[0] + 3
            opposite = positions[:, 0] > right if goal[0] < left else positions[:, 0] < left
            door = np.abs(positions[:, 0] - layout[0]) <= 10
            door &= np.abs(positions[:, 1] - layout[1]) <= 12
            index = {tuple(p): i for i, p in enumerate(positions)}
            edges = [(i, index[q]) for i, p in enumerate(positions)
                     for q in ((p[0] + 2, p[1]), (p[0], p[1] + 2))
                     if q in index and _visible(p, np.asarray(q), layout)]
            report = {}
            for label, cost in scores.items():
                local = []
                for a, b in edges:
                    if door[a] or door[b]:
                        if abs(truth[a] - truth[b]) >= GAP:
                            far, near = (a, b) if truth[a] > truth[b] else (b, a)
                            local.append(cost[far] - cost[near])
                local = np.asarray(local)
                report[label] = dict(full=_rank_metric(truth, cost),
                                     opposite=_rank_metric(truth[opposite], cost[opposite]),
                                     doorway_edges=len(local),
                                     doorway_uphill=int((local < -max(1e-7, .001 * np.ptp(cost))).sum()),
                                     self_cost=self_cost[label])
            row = dict(layout=layout, goal=goal, goal_id=goal_number,
                       valid_points=len(grid_ids), metrics=report,
                       axis=design["axis"], grid=design["grid"],
                       geodesic=truth.tolist(), straight_line=np.linalg.norm(positions - goal, axis=1).tolist(),
                       positions=positions.tolist(), costs={k: v.tolist() for k, v in scores.items()})
            write_json(output / f"{key}_goal{goal_number + 1}.json", row)
            all_rows.append(row)
    write_json(output / "summary.json", [{k: row[k] for k in ("layout", "goal", "valid_points", "metrics")}
                                         for row in all_rows])
    return all_rows


def changed_order_report(rows, *, reference_layout=None):
    """Matched query pairs whose true ordering reverses between layouts."""
    result = []
    for goal in sorted({tuple(row["goal"]) for row in rows}):
        selected = [row for row in rows if tuple(row["goal"]) == goal]
        for left, right in combinations(selected, 2):
            if reference_layout is not None and tuple(reference_layout) not in (tuple(left["layout"]), tuple(right["layout"])):
                continue
            a = {tuple(p): i for i, p in enumerate(left["positions"])}
            b = {tuple(p): i for i, p in enumerate(right["positions"])}
            common = sorted(a.keys() & b.keys())
            truth_a = np.asarray(left["geodesic"])[[a[p] for p in common]]
            truth_b = np.asarray(right["geodesic"])[[b[p] for p in common]]
            x, y = np.triu_indices(len(common), 1)
            sign_a = np.sign(truth_a[x] - truth_a[y])
            sign_b = np.sign(truth_b[x] - truth_b[y])
            flipped = (sign_a * sign_b < 0) & (np.abs(truth_a[x] - truth_a[y]) >= GAP) & (np.abs(truth_b[x] - truth_b[y]) >= GAP)
            x, y, sign_a, sign_b = x[flipped], y[flipped], sign_a[flipped], sign_b[flipped]
            models = {}
            for label in left["costs"].keys() & right["costs"].keys():
                full_a = np.asarray(left["costs"][label])
                full_b = np.asarray(right["costs"][label])
                cost_a = full_a[[a[p] for p in common]]
                cost_b = full_b[[b[p] for p in common]]
                delta_a, delta_b = cost_a[x] - cost_a[y], cost_b[x] - cost_b[y]
                tolerance_a = max(1e-7, .001 * float(np.ptp(full_a)))
                tolerance_b = max(1e-7, .001 * float(np.ptp(full_b)))
                predicted_a = np.where(np.abs(delta_a) <= tolerance_a, 0, np.sign(delta_a))
                predicted_b = np.where(np.abs(delta_b) <= tolerance_b, 0, np.sign(delta_b))
                models[label] = dict(changed=int((predicted_a * predicted_b < 0).sum()),
                                     correct_both=int(((predicted_a == sign_a) & (predicted_b == sign_b)).sum()),
                                     ties=int(((predicted_a == 0) | (predicted_b == 0)).sum()))
            result.append(dict(goal=goal, layouts=[left["layout"], right["layout"]],
                               common_legal_points=len(common), truth_flips=len(x), models=models))
    return result


In [ ]:
assert sha256(REPO/'src/mylonas_ijepa_wm/dotwall_layout_cost.py')==MODULE_SHA
sys.path.insert(0,str(REPO))
from types import SimpleNamespace
from src.mylonas_ijepa_wm import dotwall_layout_cost as transfer
from src.mylonas_ijepa_wm import plan_pointmaze as visual
from src.mylonas_ijepa_wm.curriculum_runtime import PilotRuntime
from src.mylonas_ijepa_wm.environment_adapters import WallAdapter
from src.mylonas_ijepa_wm.environment_configs import EnvironmentConfig
from src.mylonas_ijepa_wm.curriculum_protocol import write_json
encoder=visual.load_frozen_ijepa_encoder(ENCODER,torch.device('cuda'))
rt=SimpleNamespace(device=torch.device('cuda'),precision='fp32',args=SimpleNamespace(encoder_batch_size=8),load_encoder=lambda:encoder)
encode=lambda frames:PilotRuntime.encode_frames(rt,frames)
env=WallAdapter(EnvironmentConfig(env='wall',wall_layouts=[list(x) for x in transfer.ALL_LAYOUTS],dino_wm_root=str(DINO_ROOT)),100)
provenance=dict(source_commit=SOURCE_COMMIT,runner_sha256=MODULE_SHA,renderer_commit=DINO_COMMIT,encoder_sha256=ENCODER_SHA,torch_version=str(torch.__version__),device_name=torch.cuda.get_device_name())


## Smoke check

The standard layout and one additional **seen** training layout, the first square band, a sparse offset grid, and eight updates per adapter. This exercises layout-balanced fitting without exposing any held-out layout. Inspect the metrics before full fitting. Smoke output stays on the VM.


In [ ]:
smoke=transfer.prepare(Path('/content/layout-cost-smoke-output')/MODULE_SHA[:12],Path('/content/layout-cost-smoke-cache')/MODULE_SHA[:12],env,encode,runtime=provenance,smoke=True)
smoke_fixed=transfer.fit_model(smoke,'fixed',updates=8,batch_size=4,device='cuda')
smoke_multi=transfer.fit_model(smoke,'multi',updates=8,batch_size=4,device='cuda')
smoke_rows=transfer.evaluate(smoke,[transfer.layout_id(transfer.TRAIN_LAYOUTS[0])],{'fixed':smoke_fixed,'multi':smoke_multi},'seen',device='cuda')
for row in smoke_rows:print(row['layout'],row['goal'],{k:v['full'] for k,v in row['metrics'].items()})


## Full experiment

Set `RUN_FULL=True` after the smoke check. Both fits use the same eight bands, 500 updates per band, and batch size 16. The second fit starts from a fresh identity adapter and samples layouts then goals evenly. The four unseen layouts are never used for fitting or checkpoint selection.


In [ ]:
RUN_FULL=False  # Change to True, then run stages 1–4 in order.
if RUN_FULL:
    study=transfer.prepare(OUTPUT,CACHE,env,encode,runtime=provenance)
    print('Token cache:',tuple(study['tokens'].shape))
else:print('Full run disabled.')


### 1. Fixed adapter, standard layout

Fit on `(32,30)` and evaluate held-out goals and offset query positions there.


In [ ]:
if RUN_FULL:
    fixed_checkpoint=transfer.fit_model(study,'fixed',device='cuda')
    standard_rows=transfer.evaluate(study,[transfer.layout_id(transfer.STANDARD)],{'fixed':fixed_checkpoint},'standard',device='cuda')
    print(OUTPUT/'standard/summary.json')


### 2. Fixed adapter, unseen layouts

Freeze the stage 1 checkpoint. Render each moved wall and doorway and use its own geodesic labels. Save matched standard-to-unseen truth-order changes.


In [ ]:
if RUN_FULL:
    unseen_keys=[transfer.layout_id(x) for x in transfer.TEST_LAYOUTS]
    fixed_unseen=transfer.evaluate(study,unseen_keys,{'fixed':fixed_checkpoint},'fixed_unseen',device='cuda')
    fixed_changes=transfer.changed_order_report(standard_rows+fixed_unseen,reference_layout=transfer.STANDARD)
    write_json(OUTPUT/'fixed_unseen/changed_order_vs_standard.json',fixed_changes)
    print(OUTPUT/'fixed_unseen/summary.json',OUTPUT/'fixed_unseen/changed_order_vs_standard.json')


### 3. Multi-layout adapter, seen layouts

Train a new shared adapter on the nine training layouts. Each band report records actual draws by layout and goal.


In [ ]:
if RUN_FULL:
    multi_checkpoint=transfer.fit_model(study,'multi',device='cuda')
    multi_seen=transfer.evaluate(study,[transfer.layout_id(x) for x in transfer.TRAIN_LAYOUTS],{'multi':multi_checkpoint},'multi_seen',device='cuda')
    print(OUTPUT/'multi_seen/summary.json')


### 4. Multi-layout adapter, unseen layouts

Score raw MSE and both frozen adapters on the exact same unseen layout, goal, and query cases. Re-evaluate the standard layout with both adapters, then report matched standard-to-unseen truth-order reversals.


In [ ]:
if RUN_FULL:
    both_unseen=transfer.evaluate(study,unseen_keys,{'fixed':fixed_checkpoint,'multi':multi_checkpoint},'both_unseen',device='cuda')
    standard_both=transfer.evaluate(study,[transfer.layout_id(transfer.STANDARD)],{'fixed':fixed_checkpoint,'multi':multi_checkpoint},'standard_both',device='cuda')
    assert [(r['layout'],r['goal'],r['positions']) for r in fixed_unseen]==[(r['layout'],r['goal'],r['positions']) for r in both_unseen]
    assert all(a['costs']['fixed']==b['costs']['fixed'] for a,b in zip(fixed_unseen,both_unseen))
    assert all(a['positions']==b['positions'] and a['costs']['fixed']==b['costs']['fixed'] for a,b in zip(standard_rows,standard_both))
    changed_standard=transfer.changed_order_report(standard_both+both_unseen,reference_layout=transfer.STANDARD)
    changed_unseen=transfer.changed_order_report(both_unseen)
    write_json(OUTPUT/'both_unseen/changed_order_vs_standard.json',changed_standard)
    write_json(OUTPUT/'both_unseen/changed_order_between_unseen.json',changed_unseen)
    print(OUTPUT/'both_unseen/summary.json',OUTPUT/'both_unseen/changed_order_vs_standard.json')


## Reports and aligned maps

Each map has its own scale. The straight-line control shows where the wall changes legal distance. Gray blocks mark solid wall, and the star marks the goal. Reports include pair counts, ordering accuracy, Spearman correlation, opposite-room ordering, doorway uphill edges, and exact goal self cost.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd

def heatmaps(rows,folder):
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    for row in rows:
        axis=row['axis']; lookup={v:i for i,v in enumerate(axis)}
        panels=[('Geodesic',row['geodesic']),('Straight line',row['straight_line'])]
        panels += [(name,row['costs'][name]) for name in ('raw','fixed','multi') if name in row['costs']]
        fig,axes=plt.subplots(1,len(panels),figsize=(4.1*len(panels),4.4),constrained_layout=True)
        wx,dy=row['layout']
        for ax,(title,values) in zip(axes,panels):
            field=np.full((len(axis),len(axis)),np.nan)
            for (x,y),v in zip(row['positions'],values):field[lookup[y],lookup[x]]=v
            image=ax.pcolormesh(axis,axis,field,cmap='viridis',shading='nearest',vmin=0)
            for top,bottom in ((axis[0],dy-4),(dy+4,axis[-1])):
                ax.add_patch(Rectangle((wx-3,top),6,bottom-top,facecolor='#424242',zorder=3))
            ax.scatter(*row['goal'],marker='*',c='black',edgecolor='white',s=110,zorder=4)
            ax.set(title=title,xlim=(axis[0],axis[-1]),ylim=(axis[-1],axis[0]),aspect='equal',xlabel='Ball x',ylabel='Ball y')
            fig.colorbar(image,ax=ax,shrink=.78)
        fig.suptitle(f"Wall {wx}, door {dy}, goal {tuple(row['goal'])}")
        fig.savefig(folder/f"wall{wx:02d}_door{dy:02d}_goal{row['goal_id']+1}.png",dpi=130)
        plt.show();plt.close(fig)

if RUN_FULL:
    for group in ('standard_rows','fixed_unseen','multi_seen','standard_both','both_unseen'):
        heatmaps(globals()[group],OUTPUT/group/'heatmaps')
    table=pd.DataFrame([dict(layout=tuple(row['layout']),goal=tuple(row['goal']),model=model,
        pairs=m['full']['pairs'],accuracy=m['full']['accuracy'],spearman=m['full']['spearman'],
        opposite_pairs=m['opposite']['pairs'],opposite_accuracy=m['opposite']['accuracy'],
        doorway_edges=m['doorway_edges'],doorway_uphill=m['doorway_uphill'])
        for row in both_unseen for model,m in row['metrics'].items()])
    display(table)
    table.to_csv(OUTPUT/'both_unseen/metrics.csv',index=False)


## Return the run folder

Send back the Drive run folder after the full run. It contains layout and goal manifests, frame-hash digests, provenance, per-band draw reports, checkpoints, per-goal reports, standard-to-unseen and unseen-to-unseen changed-order summaries, and heatmaps. The Colab VM token cache can be discarded.
